In [65]:
import requests
import pandas as pd
import json
from dateutil import parser

In [5]:
API_KEY = "25ca3f1c80e0bd573abf125260d832d8-b9884a5701c2ca83d3a7a00e0fd3a1f0"
ACCOUNT_ID = "101-011-30426832-001"
OANDA_URL = "https://api-fxpractice.oanda.com/v3"

In [6]:
session = requests.Session()

In [8]:
session.headers.update({
    "Authorization": f"Bearer {API_KEY}",
    "Content_Type": "application/json"
})

In [19]:
params = dict(
    count = 10,
    granularity = "H1",
    price = "MBA"
)

In [23]:
url = f"{OANDA_URL}/accounts/{ACCOUNT_ID}/instruments"

In [24]:
res = session.get(url, params=None, data=None, headers=None)

In [30]:
print("Status Code:", res.status_code)
data = res.json()

Status Code: 200


In [32]:
instruments_list = data['instruments']
print("Instruments length:", len(instruments_list))

Instruments length: 127


In [33]:
instruments_list[0].keys()

dict_keys(['name', 'type', 'displayName', 'pipLocation', 'displayPrecision', 'tradeUnitsPrecision', 'minimumTradeSize', 'maximumTrailingStopDistance', 'minimumTrailingStopDistance', 'maximumPositionSize', 'maximumOrderUnits', 'marginRate', 'guaranteedStopLossOrderMode', 'tags', 'financing'])

In [34]:
key_i = ['name', 'type', 'displayName', 'pipLocation', 'displayPrecision', 'tradeUnitsPrecision', 'marginRate']

In [42]:
instruments_dict = {}
for instr in instruments_list:
    key = instr['name']
    instruments_dict[key] = { k: instr[k] for k in key_i }

instruments_dict['USD_CAD']

{'name': 'USD_CAD',
 'type': 'CURRENCY',
 'displayName': 'USD/CAD',
 'pipLocation': -4,
 'displayPrecision': 5,
 'tradeUnitsPrecision': 0,
 'marginRate': '0.0333'}

In [46]:
with open("../data/instruments.json", "w") as f:
    f.write(json.dumps(instruments_dict, indent=3))

In [83]:
def fetch_candles(pair_name, count=10, granularity="H1"):
    url = f"{OANDA_URL}/instruments/{pair_name}/candles"
    params = dict(
        count = count,
        granularity = granularity,
        price = "MBA"
    )
    res = session.get(url, params=params, data=None, headers=None)
    data = res.json()
    

    if res.status_code == 200:
        if 'candles' not in data:
            data = []
        else:
            data = data['candles']

    return res.status_code, data

def get_candles_df(data):
    if len(data) == 0:
        return pd.DataFrame()
    
    prices = ['mid', 'bid', 'ask']
    ohlc = ['o', 'h', 'l', 'c']

    final_data = []
    for candle in data:
        if candle['complete'] == False:
            continue
        new_dict = {}
        new_dict['time'] = parser.parse(candle['time'])

        new_dict['volume'] = candle['volume']
        for p in prices:
            for o in ohlc:
                new_dict[f"{p}_{o}"] = float(candle[p][o])
        final_data.append(new_dict)

    # Since the last candle in incomplete, it would not be included
    # Thus, total candles = count-1
    df = pd.DataFrame.from_dict(final_data)
    return df

def create_data_file(pair_name, count=10, granularity="H1"):
    code, data = fetch_candles(pair_name, count, granularity)
    if code != 200:
        print("Failed", pair_name, data)
        return
    if len(data) == 0:
        print("No candles", pair_name)
    candles_df = get_candles_df(data)
    candles_df.to_pickle(f"../data/{pair_name}_{granularity}.pkl")
    print(f"{pair_name} {granularity} {candles_df.shape[0]} candles, {candles_df.time.min()} {candles_df.time.max()}")

In [84]:
code, data = fetch_candles("EUR_USD", count=10, granularity="H4")
candles_df = get_candles_df(data)
candles_df

,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c
0,2024-11-22 14:00:00+00:00,50348,1.04296,1.04348,1.03920,1.04190,1.04289,1.04341,1.03912,1.04183,1.04304,1.04355,1.03928,1.04198
1,2024-11-22 18:00:00+00:00,15979,1.04190,1.04255,1.04081,1.04186,1.04182,1.04248,1.04073,1.04141,1.04198,1.04262,1.04089,1.04230
2,2024-11-24 22:00:00+00:00,23642,1.04786,1.05016,1.04634,1.04781,1.04762,1.05009,1.04619,1.04774,1.04810,1.05023,1.04648,1.04788
3,2024-11-25 02:00:00+00:00,15206,1.04778,1.04895,1.04746,1.04792,1.04771,1.04887,1.04738,1.04784,1.04786,1.04903,1.04753,1.04800
4,2024-11-25 06:00:00+00:00,40875,1.04792,1.04972,1.04490,1.04896,1.04784,1.04965,1.04483,1.04888,1.04799,1.04980,1.04497,1.04904
5,2024-11-25 10:00:00+00:00,37943,1.04896,1.05304,1.04740,1.05285,1.04888,1.05296,1.04733,1.05277,1.04903,1.05312,1.04748,1.05293
6,2024-11-25 14:00:00+00:00,46532,1.05286,1.05304,1.04667,1.04861,1.05279,1.05296,1.04660,1.04853,1.05294,1.05311,1.04674,1.04869
7,2024-11-25 18:00:00+00:00,17994,1.04861,1.05108,1.04856,1.04966,1.04854,1.05101,1.04849,1.04956,1.04868,1.05115,1.04863,1.04975
8,2024-11-25 22:00:00+00:00,36351,1.04942,1.05008,1.04250,1.04541,1.04904,1.04999,1.04242,1.04533,1.04979,1.05018,1.04258,1.04549


In [86]:
create_data_file("EUR_USD", count=10, granularity="H4")

EUR_USD H4 9 candles, 2024-11-22 14:00:00+00:00 2024-11-25 22:00:00+00:00


In [88]:
our_curr = ['EUR', 'USD', 'GBP', 'JPY', 'CHF', 'NZD', 'CAD', 'AUD']
instruments_dict.keys()

dict_keys(['XAG_SGD', 'AUD_NZD', 'BCO_USD', 'NZD_USD', 'CORN_USD', 'NL25_EUR', 'CAD_JPY', 'USD_ZAR', 'SG30_SGD', 'EUR_USD', 'SOYBN_USD', 'XAU_EUR', 'XPT_USD', 'USD_DKK', 'AU200_AUD', 'XAU_XAG', 'XAU_GBP', 'NAS100_USD', 'GBP_AUD', 'USD_PLN', 'CHINAH_HKD', 'CH20_CHF', 'CAD_HKD', 'BCH_USD', 'XAG_CHF', 'USD_CHF', 'XAG_HKD', 'AUD_HKD', 'ESPIX_EUR', 'NZD_CHF', 'AUD_CHF', 'GBP_CHF', 'USD_THB', 'XAU_JPY', 'XAU_HKD', 'EUR_HKD', 'CHF_JPY', 'GBP_HKD', 'EUR_NZD', 'XAG_AUD', 'WTICO_USD', 'XAG_NZD', 'AUD_SGD', 'EUR_JPY', 'EUR_TRY', 'USD_JPY', 'BTC_USD', 'SGD_JPY', 'GBP_ZAR', 'XAG_JPY', 'ETH_USD', 'ZAR_JPY', 'NZD_SGD', 'EUR_DKK', 'USD_HUF', 'HKD_JPY', 'DE30_EUR', 'US2000_USD', 'NATGAS_USD', 'DE10YB_EUR', 'GBP_CAD', 'UK100_GBP', 'EUR_HUF', 'USD_SEK', 'GBP_SGD', 'XPD_USD', 'XAU_CHF', 'XAU_CAD', 'EUR_PLN', 'SUGAR_USD', 'AUD_CAD', 'USB05Y_USD', 'UK10YB_GBP', 'EUR_CAD', 'USD_MXN', 'GBP_USD', 'CAD_SGD', 'XAG_CAD', 'JP225_USD', 'FR40_EUR', 'USB30Y_USD', 'NZD_HKD', 'XAG_USD', 'EUR_CZK', 'EUR_CHF', 'WHEAT_USD

In [89]:
for c1 in our_curr:
    for c2 in our_curr:
        pair = f"{c1}_{c2}"
        if pair in instruments_dict:
            for g in ["H1", "H4"]:
                create_data_file(pair, count=4001, granularity=g)

EUR_USD H1 4000 candles, 2024-04-05 10:00:00+00:00 2024-11-26 04:00:00+00:00
EUR_USD H4 4000 candles, 2022-05-03 09:00:00+00:00 2024-11-25 22:00:00+00:00
EUR_GBP H1 4000 candles, 2024-04-05 10:00:00+00:00 2024-11-26 04:00:00+00:00
EUR_GBP H4 4000 candles, 2022-05-03 09:00:00+00:00 2024-11-25 22:00:00+00:00
EUR_JPY H1 4000 candles, 2024-04-05 10:00:00+00:00 2024-11-26 04:00:00+00:00
EUR_JPY H4 4000 candles, 2022-05-03 01:00:00+00:00 2024-11-25 22:00:00+00:00
EUR_CHF H1 4000 candles, 2024-04-05 10:00:00+00:00 2024-11-26 04:00:00+00:00
EUR_CHF H4 4000 candles, 2022-05-03 09:00:00+00:00 2024-11-25 22:00:00+00:00
EUR_NZD H1 4000 candles, 2024-04-05 10:00:00+00:00 2024-11-26 04:00:00+00:00
EUR_NZD H4 4000 candles, 2022-05-03 17:00:00+00:00 2024-11-25 22:00:00+00:00
EUR_CAD H1 4000 candles, 2024-04-05 10:00:00+00:00 2024-11-26 04:00:00+00:00
EUR_CAD H4 4000 candles, 2022-05-03 09:00:00+00:00 2024-11-25 22:00:00+00:00
EUR_AUD H1 4000 candles, 2024-04-05 10:00:00+00:00 2024-11-26 04:00:00+00:00